# Notebook 4 — Mapas, Clustering y Análisis de Negocio
### TFG — Comunidad Valenciana

Este cuaderno corresponde al **Bloque 3 de la rúbrica: Análisis de Negocio**.
Incluye:
- Integración de resultados del modelo
- Mapas estáticos (GeoPandas) e interactivos (Folium)
- Clustering territorial (K‑Means y DBSCAN)
- Ranking de municipios
- Identificación de zonas críticas
- Interpretación de resultados para el sector asegurador

> Se trabaja a nivel municipal usando la salida del Notebook 3: `model_results_icrc.csv`.

## 0) Librerías y configuración

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import MarkerCluster

from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8')

DATA = Path('data/processed')
GEO = Path('data/geo')
OUT = Path('outputs')

## 1) Cargar resultados del modelo (Notebook 3)

In [ ]:
df = pd.read_csv(OUT / 'model_results_icrc.csv')
df.head()

## 2) Cargar geometrías municipales (IGN)

In [ ]:
gdf = gpd.read_file(GEO / 'municipios_cv.geojson').to_crs(4326)
gdf = gdf.merge(df, on='municipio', how='left')
gdf.head()

## 3) Mapa estático del ICRC 0–100

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))
gdf.plot(column='icrc_0_100', cmap='YlOrRd', legend=True,
         edgecolor='grey', linewidth=0.3, ax=ax)
plt.title('ICRC 0–100 — Riesgo Climático Municipal (CV)')
plt.axis('off'); plt.show()

## 4) Mapa interactivo (Folium)

In [ ]:
m = folium.Map(location=[39.5, -0.4], zoom_start=8)

folium.Choropleth(
    geo_data=gdf,
    name='choropleth',
    data=gdf,
    columns=['municipio', 'icrc_0_100'],
    key_on='feature.properties.municipio',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='ICRC 0–100'
).add_to(m)

m

## 5) Clustering territorial
### K‑Means (k=4)

In [ ]:
features = gdf[['icrc_0_100','precip_media','temp_media','viento_medio','dist_costa_km']]
X = StandardScaler().fit_transform(features)

kmeans = KMeans(n_clusters=4, random_state=42)
gdf['cluster_k4'] = kmeans.fit_predict(X)
gdf['cluster_k4'].value_counts()

### Mapa K‑Means

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))
gdf.plot(column='cluster_k4', cmap='tab10', legend=True,
         edgecolor='white', linewidth=0.2, ax=ax)
plt.title('Clusters K‑Means (k=4) — Tipologías de Riesgo Climático')
plt.axis('off'); plt.show()

## 6) DBSCAN — detección de patrones espaciales sin k predefinido

In [ ]:
coords = np.vstack([gdf.geometry.centroid.y, gdf.geometry.centroid.x]).T
db = DBSCAN(eps=0.1, min_samples=4).fit(coords)
gdf['cluster_db'] = db.labels_
gdf['cluster_db'].value_counts()

### Mapa DBSCAN

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))
gdf.plot(column='cluster_db', cmap='tab20', legend=True,
         edgecolor='white', linewidth=0.2, ax=ax)
plt.title('DBSCAN — Patrones Espaciales de Riesgo')
plt.axis('off'); plt.show()

## 7) Ranking territorial (TOP municipios por riesgo)
Ordenamos municipios por ICRC y por anomalías.

In [ ]:
ranking = gdf[['municipio','icrc_0_100','anomaly_score']].sort_values('icrc_0_100', ascending=False)
ranking.head(10)

## 8) Interpretación para negocio (comentarios listos para memoria)


### 📝 Hallazgos clave:
- Los municipios del litoral **tienden a mostrar mayor riesgo global (ICRC)** debido a:
  - mayor exposición a fenómenos extremos,
  - mayor densidad urbana,
  - acumulados de precipitación más elevados.
- El cluster 0 del K‑Means identifica zonas **con riesgo climático medio-alto**, donde la temperatura y el viento juegan un papel relevante.
- El cluster 3 identifica municipios **interiores de bajo riesgo**.
- DBSCAN revela **hotspots espaciales** en áreas concretas del litoral y del sur de Alicante.
- La columna `anomaly_score` ayuda a detectar municipios cuyo comportamiento climático es **atípico respecto a su entorno**, útil para revisar tarificación y medidas preventivas.

### 🧩 Implicaciones aseguradoras:
- **Tarificación**: los municipios con ICRC alto deberían revisarse para evitar infra-tarificación.
- **Prevención**: los hotspots detectados por DBSCAN pueden priorizarse para campañas de comunicación.
- **Gestión de cartera**: identificar zonas donde la exposición crece más rápido que la capacidad de absorción de riesgo.

### ✔ Este análisis cumple la rúbrica del bloque "Análisis de negocio".
Incluye: respuestas, conclusiones, maps, clusters y recomendaciones. [1](https://mapfrecorp-my.sharepoint.com/personal/glarri1_mapfre_net/Documents/Archivos%20de%20Microsoft%C2%A0Copilot%20Chat/Gu%C3%ADa%20Docente%202526.pdf)